# inject-detect 분류기 학습 (Colab T4)

`training/train.py` 를 그대로 호출하는 실행용 노트북이다. 학습 로직은 스크립트에만 있다.

1. 런타임 → 런타임 유형 변경 → **T4 GPU**
2. 위에서부터 차례로 실행
3. 결과는 `training/outputs/<run_name>/` 에 생기고, 마지막 셀에서 Google Drive 로 복사할 수 있다

설정은 `training/configs/*.yaml`, 실험 중 일부만 바꿀 때는 `--set key=value` 를 쓴다.

In [ ]:
# 0) 실행 설정
# 비공개 레포면 토큰 포함 URL 또는 Colab 보안 비밀을 사용
REPO_URL = "https://github.com/rusa1212/inject-detect.git"
BRANCH = "dev"
CONFIG = "training/configs/mdeberta-v3-base.yaml"  # 또는 training/configs/klue-roberta-base.yaml
OVERRIDES = []  # 예: ["learning_rate=3e-5", "epochs=5", "class_weight=balanced"]

In [ ]:
# 1) GPU 확인
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2) 레포 받기 + 의존성 설치
# Colab 에 이미 있는 torch·transformers 는 그대로 쓰고,
# 데이터 빌드·토크나이저 의존성만 추가 설치한다
import os

if not os.path.exists("inject-detect"):
    !git clone --branch {BRANCH} {REPO_URL} inject-detect
%cd inject-detect
!git log --oneline -n 1
!pip install -q -e ".[data]" sentencepiece protobuf
import torch
import transformers

print(
    "torch",
    torch.__version__,
    "| transformers",
    transformers.__version__,
    "| cuda",
    torch.cuda.is_available(),
)

In [ ]:
# 3) 데이터셋 생성 (원본 다운로드 → 통합 → 분할, data/processed/)
!python -m data.build
!head -c 600 data/processed/stats.json

In [ ]:
# 4) 학습
set_args = ("--set " + " ".join(OVERRIDES)) if OVERRIDES else ""
!python -m training.train --config {CONFIG} {set_args}

In [ ]:
# 5) 결과 확인
import json
import pathlib

import yaml

cfg = yaml.safe_load(pathlib.Path(CONFIG).read_text(encoding="utf-8"))
run_dir = pathlib.Path("training/outputs") / cfg["run_name"]
summary = json.loads((run_dir / "summary.json").read_text(encoding="utf-8"))
print("best epoch", summary["best_epoch"], summary["best_metric"], summary["best_value"])
for line in (run_dir / "history.jsonl").read_text(encoding="utf-8").splitlines():
    row = json.loads(line)
    print(row["epoch"], row["train_loss"], json.dumps(row["valid"], ensure_ascii=False))

In [ ]:
# 6) (선택) Google Drive 로 결과 복사
from google.colab import drive

drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/inject-detect
!cp -r {run_dir} /content/drive/MyDrive/inject-detect/